# Part 4 – LLM-Powered Feature

## Track C – Model Prediction Explanation Pipeline

**Name:** [Your Name]

This notebook demonstrates:
- Loading the trained model from Part 3
- Making predictions on new records
- Calling an LLM via OpenRouter API
- Validating JSON responses against a schema
- Testing a PII guardrail
- Comparing temperature 0 vs 0.7 outputs
- Saving results to CSV

Run all cells top to bottom. Requires `best_model.pkl` and a `.env` file with `LLM_API_KEY` in this folder.

## Step 1: Imports

In [ ]:
import os
import json
import requests
import pandas as pd
import joblib
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

print("Libraries imported")

## Step 2: Import Project Modules

These come from `prompts.py`, `utils.py`, and `schema.py` already in this repo.

In [ ]:
from prompts import SYSTEM_PROMPT, USER_TEMPLATE
from utils import has_pii, validate_response, fallback
from schema import SCHEMA

print("Project modules imported")

## Step 3: Load the Model

In [ ]:
model = joblib.load("best_model.pkl")
print("Model loaded from best_model.pkl")

## Step 4: API Configuration

In [ ]:
OPENROUTER_API_KEY = os.getenv("LLM_API_KEY")
OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

if not OPENROUTER_API_KEY:
    raise ValueError(
        "LLM_API_KEY not found. Create a .env file with: LLM_API_KEY=your_key"
    )

print("API key loaded from environment")

## Step 5: call_llm() Function

In [ ]:
def call_llm(prompt_text, temperature=0):
    """
    Call OpenRouter LLM with a structured prompt.

    Args:
        prompt_text (str): the user prompt
        temperature (float): 0 = deterministic, 0.7 = creative

    Returns:
        str or None: LLM response text
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    data = {
        "model": "openai/gpt-3.5-turbo",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text},
        ],
        "temperature": temperature,
        "max_tokens": 500,
    }

    try:
        response = requests.post(OPENROUTER_API_URL, headers=headers, json=data, timeout=30)
        response.raise_for_status()
        result = response.json()
        if "choices" in result and len(result["choices"]) > 0:
            return result["choices"][0]["message"]["content"]
        print("Error: unexpected response format")
        return None
    except requests.exceptions.RequestException as e:
        print(f"API Error: {e}")
        return None

print("call_llm() defined")

## Step 6: Test LLM Connectivity

In [ ]:
test_response = call_llm("Reply with only the word: hello", temperature=0)

if test_response:
    print("LLM connection OK")
    print("Response:", test_response.strip())
else:
    print("LLM connection failed")

## Step 7: Test PII Guardrail

In [ ]:
blocked_input = "Email: abc@gmail.com\nAge: 40"
print("Blocked input flagged:", has_pii(blocked_input))

allowed_input = "Age: 40\nBMI: 31\nSmoker: Yes"
print("Allowed input flagged:", has_pii(allowed_input))

## Step 8: Load Test Records

Loaded from `sample_predictions.json` (one-hot encoded, matching the Part 3 feature schema).

In [ ]:
with open("sample_predictions.json", "r") as f:
    test_records = json.load(f)

print(f"Loaded {len(test_records)} test records")
test_records

## Step 9: Helper Functions

In [ ]:
FEATURE_ORDER = [
    "age", "sex_male", "bmi", "children", "smoker_yes",
    "region_northwest", "region_southeast", "region_southwest",
]

def make_prediction(model, record):
    values = [[record[name] for name in FEATURE_ORDER]]
    try:
        prediction = model.predict(values)[0]
        probability = model.predict_proba(values)[0][1]
        return prediction, probability
    except Exception as e:
        print(f"Prediction error: {e}")
        return None, None

def get_region(record):
    if record.get("region_northwest", 0) == 1:
        return "Northwest"
    if record.get("region_southeast", 0) == 1:
        return "Southeast"
    if record.get("region_southwest", 0) == 1:
        return "Southwest"
    return "Northeast"

def format_features(record):
    return (
        f"Age: {record.get('age')}\n"
        f"Sex: {'Male' if record.get('sex_male') == 1 else 'Female'}\n"
        f"BMI: {record.get('bmi')}\n"
        f"Children: {record.get('children')}\n"
        f"Smoker: {'Yes' if record.get('smoker_yes') == 1 else 'No'}\n"
        f"Region: {get_region(record)}"
    )

print("Helper functions defined")

## Step 10: Run Predictions with LLM Explanations

In [ ]:
all_results = []

for idx, record in enumerate(test_records, 1):
    print(f"\nRecord {idx}")

    prediction, probability = make_prediction(model, record)
    if prediction is None:
        print("  Prediction failed, skipping")
        continue

    prediction_label = "High Charges" if prediction == 1 else "Low Charges"
    print(f"  Prediction: {prediction_label} (p={probability:.2%})")

    record_text = json.dumps(record)
    if has_pii(record_text):
        print("  PII detected, skipping LLM call")
        continue

    user_prompt = USER_TEMPLATE.format(
        features=format_features(record),
        prediction=prediction_label,
        probability=f"{probability:.2%}",
    )

    response_text = call_llm(user_prompt, temperature=0)
    result = validate_response(response_text) if response_text else fallback()

    validation_status = "PASS" if all(v is not None for v in result.values()) else "FAIL"
    print(f"  Validation: {validation_status}")

    result["record_id"] = idx
    result["prediction"] = int(prediction)
    result["probability"] = f"{probability:.4f}"
    result["validation_status"] = validation_status
    all_results.append(result)

print(f"\nProcessed {len(all_results)} records")

## Step 11: Temperature Comparison (0 vs 0.7)

In [ ]:
temperature_rows = []

if test_records:
    record = test_records[0]
    prediction, probability = make_prediction(model, record)
    prediction_label = "High Charges" if prediction == 1 else "Low Charges"

    user_prompt = USER_TEMPLATE.format(
        features=format_features(record),
        prediction=prediction_label,
        probability=f"{probability:.2%}",
    )

    for temp in [0, 0.7]:
        print(f"\nTemperature: {temp}")
        response = call_llm(user_prompt, temperature=temp)
        if response:
            print("  Raw (first 200 chars):", response[:200])
            parsed = validate_response(response)
            temperature_rows.append({
                "temperature": temp,
                "prediction_label": parsed.get("prediction_label"),
                "confidence_level": parsed.get("confidence_level"),
                "raw_response": response,
            })
        else:
            print("  Failed to get response")
            temperature_rows.append({
                "temperature": temp,
                "prediction_label": None,
                "confidence_level": None,
                "raw_response": None,
            })

## Step 12: Guardrail Test Log

In [ ]:
guardrail_rows = [
    {"input": "abc@gmail.com", "contains_pii": has_pii("abc@gmail.com"), "llm_called": not has_pii("abc@gmail.com")},
    {"input": "Age 45 BMI 29.2", "contains_pii": has_pii("Age 45 BMI 29.2"), "llm_called": not has_pii("Age 45 BMI 29.2")},
    {"input": "Call me at 9876543210", "contains_pii": has_pii("Call me at 9876543210"), "llm_called": not has_pii("Call me at 9876543210")},
]

for row in guardrail_rows:
    print(row)

## Step 13: Save All Outputs to CSV

In [ ]:
outputs_folder = Path("outputs")
outputs_folder.mkdir(exist_ok=True)

if all_results:
    pd.DataFrame(all_results).to_csv(outputs_folder / "llm_results.csv", index=False)
    print("Saved outputs/llm_results.csv")

if all_results:
    validation_df = pd.DataFrame([
        {"record_id": r["record_id"], "validation_status": r["validation_status"]}
        for r in all_results
    ])
    validation_df.to_csv(outputs_folder / "validation_results.csv", index=False)
    print("Saved outputs/validation_results.csv")

if temperature_rows:
    pd.DataFrame(temperature_rows).to_csv(outputs_folder / "temperature_comparison.csv", index=False)
    print("Saved outputs/temperature_comparison.csv")

pd.DataFrame(guardrail_rows).to_csv(outputs_folder / "guardrail_results.csv", index=False)
print("Saved outputs/guardrail_results.csv")

## Step 14: Summary

In [ ]:
print("PART 4 COMPLETE")
print(f"Records processed: {len(all_results)}")
print(f"Temperature comparisons: {len(temperature_rows)}")
print("All four output CSVs written to outputs/")